---
## Section 3.2 — Zero-Shot MATH Baseline

### Problem `math_baseline`

**(a)** Write a script to evaluate Qwen 2.5 Math 1.5B zero-shot performance on MATH. This script
should (1) load the MATH validation examples from /data/a5-alignment/MATH/validation.jsonl,
(2) format them as string prompts to the language model using the r1_zero prompt, and (3) gen-
erate outputs for each example. This script should also (4) calculate evaluation metrics and
(5) serialize the examples, model generations, and corresponding evaluation scores to disk for
analysis in subsequent problems.

**(b)** Run your evaluation script on Qwen 2.5 Math 1.5B. How many model generations fall into each of the following categories: (1) correct with both format and answer reward 1, (2) format reward 1 and answer reward 0, (3) format reward 0 and answer reward 0? Observing at least 10 cases where format reward is 0, do you think the issue is with the base model’s output, or the parser? Why? What about in (at least 10) cases where format reward is 1 but answer reward is 0?

**(c)** How well does the Qwen 2.5 Math 1.5B zero-shot baseline perform on MATH?

In [39]:
import os, glob
from pathlib import Path

REPO_URL = "https://github.com/bushuyeu/LLM-from-scratch.git"
REPO_DIR = Path("/content/LLM-from-scratch")

if Path("/content").exists():
    if REPO_DIR.exists():
        os.system(f"git -C {REPO_DIR} pull --ff-only -q")
    else:
        os.system(f"git clone -q {REPO_URL} {REPO_DIR}")

def _find_results_dir(name: str) -> Path | None:
    candidates = [
        Path(f"results/{name}"),
        Path(f"../results/{name}"),
        Path(f"ece405_assignment3/results/{name}"),
        Path.home() / f"projects/LLM-from-scratch/ece405_assignment3/results/{name}",
    ]
    for c in candidates:
        if c.is_dir():
            return c.resolve()
    for p in glob.glob(f"/content/**/results/{name}", recursive=True):
        if Path(p).is_dir():
            return Path(p).resolve()
    return None

MATH_BASELINE_DIR = _find_results_dir("math_baseline")
if MATH_BASELINE_DIR is None:
    raise FileNotFoundError("Cannot locate results/math_baseline after setup.")
print(f"Results at: {MATH_BASELINE_DIR}")

Results at: /content/LLM-from-scratch/ece405_assignment3/results/math_baseline


#### (a) Evaluation script

[`ece405_assignment3/cs336_alignment/math_baseline.py`](../cs336_alignment/math_baseline.py)

#### (b) Category analysis

In [17]:
import json, random
from collections import Counter, defaultdict

with open(MATH_BASELINE_DIR / "generations.jsonl") as f:
    results = [json.loads(line) for line in f if line.strip()]

by_cat = defaultdict(list)
for r in results:
    by_cat[r["category"]].append(r)

counts = Counter(r["category"] for r in results)
print(f"Total examples: {len(results)}\n")
print("Category breakdown:")
for cat, n in counts.most_common():
    print(f"  {cat:45s}  {n:5d}  ({n/len(results):.1%})")

def show_examples(category, n=10, seed=0):
    sample = random.Random(seed).sample(by_cat[category], min(n, len(by_cat[category])))
    for i, ex in enumerate(sample, 1):
        print(f"\n--- {i}. ---")
        print("RESPONSE:", ex["response"][:500])
        print(f"format_reward={ex['format_reward']}  answer_reward={ex['answer_reward']}")
        print("-" * 70)

print("\n\n=== format_wrong (format_reward=0) — 10 examples ===")
show_examples("format_wrong", n=10)

print("\n\n=== format_correct_answer_wrong (format=1, answer=0) — 10 examples ===")
show_examples("format_correct_answer_wrong", n=10)

print("\n\n=== format_correct_answer_correct (format=1, answer=1) — 5 examples ===")
show_examples("format_correct_answer_correct", n=5)

Total examples: 5000

Category breakdown:
  format_wrong                                    4290  (85.8%)
  format_correct_answer_wrong                      581  (11.6%)
  format_correct_answer_correct                    129  (2.6%)


=== format_wrong (format_reward=0) — 10 examples ===

--- 1. ---
RESPONSE: Now,
 $50\text{ }s\text{ }=20,\text{ }60\text{ }s\text{ }=\text{ }10,\text{ }70\text{ }s\text{ }=\text{ }2,\text{ }80\text{ }s\text{ }=\text{ }2$ and $\text{ }90\text{ }s\text{ }=\text{ }6$ 
</think>
<answer> $57.7$ 
</answer>
format_reward=0.0  answer_reward=0.0
----------------------------------------------------------------------

--- 2. ---
RESPONSE: Now, $473_{10}$ is $711_{8}$. Thus, the first digit is $\boxed{7}$.</think>
<answer>7</answer>
format_reward=0.0  answer_reward=0.0
----------------------------------------------------------------------

--- 3. ---
RESPONSE:  To find the total amount of money, we first need to calculate the amount each person has and then add them 

**format_reward = 0 cases (85.8% of examples):**

The base model rarely produces the required `<answer>...</answer>` structure. Common failure modes:

- **Truncation**: Many responses hit the `max_tokens=1024` limit mid-reasoning and never emit `<answer>`.
- **LaTeX `\boxed{}` instead of XML tags**: The base model was pretrained to output `\boxed{answer}` and has no signal to use the `<answer>` schema.
- **Malformed XML**: Occasionally the model produces `</think>` but wraps its answer in `<answer>$val$<br></answer>` (embedded HTML) or uses garbled tags like `<-answer/>` that fail the format check.
- **Continues past the stop token**: Some responses emit `<answer>` embedded in longer output without triggering the `</answer>` stop string.

Qwen 2.5 Math 1.5B (base, not instruct) was not trained on the `<think>…</think><answer>…</answer>` format — zero-shot compliance is low.

**format_reward = 1, answer_reward = 0 cases (11.6% of examples):**

When the model does emit a properly-formed `<answer>…</answer>` block, it still gets the answer wrong ~82% of the time. Observed patterns:

- **Arithmetic mistakes**: Correct strategy, wrong computation (e.g., correct formula, mis-evaluated intermediate step).
- **Wrong strategy**: Applies a valid method to the wrong subproblem or misreads the question.
- **Garbled content inside valid tags**: Occasionally outputs an English description (e.g., `"answer here"`) inside `<answer>` tags, which doesn't match the expected form.

**format_reward = 1, answer_reward = 1 cases (2.6% of examples):**

When both rewards are 1, the model produces clean `<think>…</think><answer>value</answer>` output and gets the answer right. 

Sampled examples show these tend to be straightforward formula application (e.g., completing the square, distance formula, prime counting) where the model's reasoning chain is short and correct.

#### (c) Deliverable — zero-shot baseline performance

Qwen 2.5 Math 1.5B achieves 2.6% accuracy (129/5000) with 14.2% format compliance zero-shot on the MATH validation set. The dominant failure mode (85.8% of examples) is that the base model never produces the required `<answer>…</answer>` XML structure, since it was pretrained to emit `\boxed{}` answers.

---
## Section 4 — Supervised Finetuning for MATH

### Problem `tokenize_prompt_and_output`

Implement a method `tokenize_prompt_and_output` that tokenizes the question and
output separately, concatenates them together, and constructs a `response_mask`.

Implemented in [`cs336_alignment/sft.py`](../cs336_alignment/sft.py) — `tokenize_prompt_and_output`.

### Problem `compute_entropy`

`compute_entropy(logits)` → per-token entropy over vocab dim, using logsumexp for numerical stability.

**Deliverable:** implementation. Test: `uv run pytest -k test_compute_entropy`

Implemented in [`cs336_alignment/sft.py`](../cs336_alignment/sft.py) — `compute_entropy`.

### Problem `get_response_log_probs` 

`get_response_log_probs(model, input_ids, labels, return_token_entropy=False)` → `{"log_probs": ..., "token_entropy": ...}`.

**Deliverable:** implementation. Test: `uv run pytest -k test_get_response_log_probs`

Implemented in [`cs336_alignment/sft.py`](../cs336_alignment/sft.py) — `get_response_log_probs`.

### Problem `masked_normalize`

`masked_normalize(tensor, mask, normalize_constant, dim=None)` — sum masked elements along dim, divide by normalize_constant.

**Deliverable:** implementation. Test: `uv run pytest -k test_masked_normalize`

Implemented in [`cs336_alignment/sft.py`](../cs336_alignment/sft.py) — `masked_normalize`.

### Problem `sft_microbatch_train_step`

`sft_microbatch_train_step(policy_log_probs, response_mask, gradient_accumulation_steps, normalize_constant=1.0)` — NLL loss on response tokens, divides by `gradient_accumulation_steps`, calls `loss.backward()`, returns `(loss, metadata)`.

**Deliverable:** implementation. Test: `uv run pytest -k test_sft_microbatch_train_step`

Implemented in [`cs336_alignment/sft.py`](../cs336_alignment/sft.py) — `sft_microbatch_train_step`.

### Problem `log_generations` — 1 point

`log_generations(...)` — log prompt, response, ground truth, reward (format/answer/total), average token entropy, and avg response length (overall, correct, incorrect).

**Deliverable:** implementation.

Implemented in [`cs336_alignment/sft.py`](../cs336_alignment/sft.py) — `log_generations`.

### Problem `sft_experiment` — 2 points

1. Run SFT on `sft.jsonl` with dataset sizes `{128, 256, 512, 1024, full}`. Tune lr/batch to ≥15% validation accuracy on full data. **Deliverable:** validation accuracy curves vs dataset size.
2. Filter SFT examples to only correct-answer ones; rerun SFT on the filtered full set. **Deliverable:** filtered dataset size + validation accuracy curve.

ECE405 deviation #6: ~30 min training budget.

In [ ]:

# ── SFT experiment: shared config ────────────────────────────────────────────
import json, subprocess, sys
from pathlib import Path
import matplotlib.pyplot as plt

pkg_root     = MATH_BASELINE_DIR.parents[1]
SFT_DATA     = Path("/data/a5-alignment/MATH/sft.jsonl")
VAL_DATA     = Path("/data/a5-alignment/MATH/validation.jsonl")
SFT_RESULTS  = pkg_root / "results" / "sft"
SFT_RESULTS.mkdir(parents=True, exist_ok=True)
TRAIN_SCRIPT = pkg_root / "cs336_alignment" / "train_sft.py"

if str(pkg_root) not in sys.path:
    sys.path.insert(0, str(pkg_root))


def load_val_curve(out_dir: Path):
    """Return (steps, accuracy_pct) from val_step_*.jsonl files."""
    steps, accs = [], []
    for p in sorted(out_dir.glob("val_step_*.jsonl")):
        step = int(p.stem.split("_")[-1])
        rows = [json.loads(l) for l in p.read_text().splitlines() if l.strip()]
        acc  = sum(r.get("reward", 0) > 0 for r in rows) / len(rows) if rows else 0.0
        steps.append(step)
        accs.append(acc * 100)
    return steps, accs


def run_sft(data_path, out_dir, *, limit=None, n_steps=500, lr=5e-5,
            grad_accum=8, no_wandb=False, tag=""):
    """Launch train_sft.py; streams stdout live. Skips if results exist."""
    out_dir = Path(out_dir)
    if list(out_dir.glob("val_step_*.jsonl")):
        print("[skip] " + out_dir.name + " already has results.")
        return
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable, str(TRAIN_SCRIPT),
        "--model",                       "Qwen/Qwen2.5-Math-1.5B",
        "--data-path",                   str(data_path),
        "--val-data-path",               str(VAL_DATA),
        "--output-dir",                  str(out_dir),
        "--n-steps",                     str(n_steps),
        "--learning-rate",               str(lr),
        "--micro-batch-size",            "1",
        "--gradient-accumulation-steps", str(grad_accum),
        "--val-every",                   "50",
        "--val-limit",                   "256",
        "--gpu-memory-utilization",      "0.45",
        "--wandb-project",               "ece405-sft",
        "--wandb-run-name",              tag or ("sft-limit" + str(limit or "full") + "-lr" + str(lr)),
    ]
    if limit is not None:
        cmd += ["--limit", str(limit)]
    if no_wandb:
        cmd += ["--no-wandb"]
    label = tag or ("limit=" + str(limit or "full"))
    print("\n" + "="*60 + "\n  " + label + "\n" + "="*60 + "\n")
    subprocess.run(cmd, cwd=str(pkg_root), check=True)


print("Config ready.")
print("  pkg_root    : " + str(pkg_root))
print("  SFT_DATA    : " + str(SFT_DATA))
print("  SFT_RESULTS : " + str(SFT_RESULTS))


In [ ]:

# ── Experiment 1: dataset size sweep (~30 min × 5 runs on T4) ────────────────
# Hyperparams tuned for ≥15 % val accuracy on full dataset:
#   lr=5e-5, effective batch = 1 × 8 grad_accum = 8 examples, 500 opt steps
SWEEP_LIMITS = [128, 256, 512, 1024, None]   # None = full dataset

for limit in SWEEP_LIMITS:
    tag = f"sweep-{limit or 'full'}"
    run_sft(
        SFT_DATA,
        SFT_RESULTS / f"sweep_{limit or 'full'}",
        limit=limit,
        n_steps=500,
        lr=5e-5,
        tag=tag,
    )

print("\nAll sweep runs finished (or were already complete).")


In [ ]:

# ── Plot: dataset size sweep accuracy curves ──────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

for limit in SWEEP_LIMITS:
    tag  = limit or "full"
    out  = SFT_RESULTS / f"sweep_{tag}"
    steps, accs = load_val_curve(out)
    if not steps:
        print(f"No results yet for limit={tag}")
        continue
    label = f"n = {limit:,}" if limit else "full dataset"
    ax.plot(steps, accs, marker="o", label=label)

ax.axhline(15, color="gray", linestyle="--", linewidth=0.8, label="15 % target")
ax.set_xlabel("Optimizer steps")
ax.set_ylabel("Validation accuracy (%)")
ax.set_title("SFT: Validation accuracy vs dataset size (Qwen2.5-Math-1.5B, lr=5e-5)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(SFT_RESULTS / "sweep_curves.png", dpi=150)
plt.show()


#### Experiment 2 — Filtered dataset (correct examples only)

In [ ]:

# ── Inspect SFT data format (no GPU / grader deps needed) ────────────────────
FILTERED_PATH = SFT_RESULTS / "sft_filtered.jsonl"

all_examples = [json.loads(l) for l in SFT_DATA.read_text().splitlines() if l.strip()]
ex0 = all_examples[0]

print(f"Total SFT examples : {len(all_examples):,}")
print(f"Fields             : {list(ex0.keys())}")
print()
for k, v in ex0.items():
    print(f"  {k:25s}: {str(v)[:180]}")

# Detect which field holds the correct answer
_GT_CANDIDATES = ["answer", "ground_truth", "gt_answer", "expected_answer"]
_GT_FIELD = next((f for f in _GT_CANDIDATES if f in ex0), None)
print(f"\nGround truth field : {_GT_FIELD!r}",
      "(auto-detected)" if _GT_FIELD else "⚠ NOT FOUND — update filter cell manually")


In [ ]:

# ── Filter SFT data to correct-answer examples only ───────────────────────────
# Installs grader deps if needed (idempotent).
import subprocess as _sp, sys as _sys
_sp.run([_sys.executable, "-m", "pip", "install", "-q",
         "latex2sympy2_extended", "math_verify", "pylatexenc"], check=False)

from cs336_alignment.drgrpo_grader import r1_zero_reward_fn  # noqa: E402

if FILTERED_PATH.exists():
    print(f"Filtered file already exists: {FILTERED_PATH}")
    filtered = [json.loads(l) for l in FILTERED_PATH.read_text().splitlines() if l.strip()]
else:
    # _GT_FIELD was detected by the inspection cell above.
    # If it printed ⚠ NOT FOUND, set it manually here, e.g. _GT_FIELD = "answer"
    filtered = []
    for ex in all_examples:
        solution = ex.get("solution") or ex.get("response") or ""
        gt = ex.get(_GT_FIELD, "") if _GT_FIELD else ""
        if r1_zero_reward_fn(solution, gt).get("reward", 0) > 0:
            filtered.append(ex)

    with open(FILTERED_PATH, "w") as f:
        for ex in filtered:
            f.write(json.dumps(ex) + "\n")

print(f"Filtered : {len(filtered):,} / {len(all_examples):,} examples "
      f"({len(filtered)/len(all_examples):.1%} correct)")
print(f"Saved to : {FILTERED_PATH}")


In [ ]:

# ── Experiment 2: SFT on filtered (correct-only) full dataset (~30 min) ───────
run_sft(
    FILTERED_PATH,
    SFT_RESULTS / "filtered_full",
    limit=None,
    n_steps=500,
    lr=5e-5,
    tag="sft-filtered-full",
)
print("Filtered-dataset run complete.")


In [ ]:

# ── Plot: filtered vs unfiltered full dataset ─────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

for label, out in [
    ("full dataset (all examples)",   SFT_RESULTS / "sweep_full"),
    ("full dataset (correct only)",   SFT_RESULTS / "filtered_full"),
]:
    steps, accs = load_val_curve(out)
    if steps:
        ax.plot(steps, accs, marker="o", label=label)
    else:
        print(f"No results yet for: {out.name}")

ax.axhline(15, color="gray", linestyle="--", linewidth=0.8, label="15 % target")
ax.set_xlabel("Optimizer steps")
ax.set_ylabel("Validation accuracy (%)")
ax.set_title("SFT: Full dataset vs correct-only filtered (Qwen2.5-Math-1.5B)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(SFT_RESULTS / "filtered_vs_full.png", dpi=150)
plt.show()


---
## Section 5 — Expert Iteration for MATH

EI: sample G rollouts per question, keep correct ones, SFT on them, repeat for `n_ei_steps`.

### Problem `expert_iteration_experiment` — 2 points

Run EI on MATH with `n_ei_steps=5`. Vary G ∈ rollouts and epochs in SFT step (≥2 configs each). Batch size `D_b` ∈ `{512, 1024, 2048}`.

**Deliverables:** validation accuracy curves per config; model ≥15% accuracy; 2-sentence comparison vs SFT; entropy plot over training.

In [ ]:
# TODO: run expert iteration and display results

---
## Section 7 — Group Relative Policy Optimization

GRPO: sample G outputs per question, group-normalize advantages, off-policy update with PPO-style clipping. Tests: `tests/test_grpo.py`.

### Problem `compute_group_normalized_rewards` — 2 points

`compute_group_normalized_rewards(reward_fn, rollout_responses, repeated_ground_truths, group_size, advantage_eps, normalize_by_std)` → `(advantages, raw_rewards, metadata)`. If `normalize_by_std=False`, advantage is $r - \bar{r}$ (Dr. GRPO).

**Deliverable:** implementation. Test: `uv run pytest -k test_compute_group_normalized_rewards`

Implemented in [`cs336_alignment/grpo.py`](../cs336_alignment/grpo.py) — `compute_group_normalized_rewards`.

### Problem `compute_naive_policy_gradient_loss` — 1 point

`compute_naive_policy_gradient_loss(raw_rewards_or_advantages, policy_log_probs)` → per-token loss $-A_t \cdot \log \pi(o_t|\ldots)$.

**Deliverable:** implementation. Test: `uv run pytest -k test_compute_naive_policy_gradient_loss`

Implemented in [`cs336_alignment/grpo.py`](../cs336_alignment/grpo.py) — `compute_naive_policy_gradient_loss`.

### Problem `compute_grpo_clip_loss` — 2 points

`compute_grpo_clip_loss(advantages, policy_log_probs, old_log_probs, cliprange)` → clipped PG loss + clip-fraction metadata.

**Deliverable:** implementation. Test: `uv run pytest -k test_compute_grpo_clip_loss`

Implemented in [`cs336_alignment/grpo.py`](../cs336_alignment/grpo.py) — `compute_grpo_clip_loss`.

### Problem `compute_policy_gradient_loss` — 1 point

Wrapper dispatching on `loss_type ∈ {no_baseline, reinforce_with_baseline, grpo_clip}`.

**Deliverable:** implementation. Test: `uv run pytest -k test_compute_policy_gradient_loss`

Implemented in [`cs336_alignment/grpo.py`](../cs336_alignment/grpo.py) — `compute_policy_gradient_loss`.

### Problem `masked_mean` — 1 point

`masked_mean(tensor, mask, dim=None)` → mean over masked positions.

**Deliverable:** implementation. Test: `uv run pytest -k test_masked_mean`

Implemented in [`cs336_alignment/grpo.py`](../cs336_alignment/grpo.py) — `masked_mean`.

### Problem `grpo_microbatch_train_step` — 3 points

`grpo_microbatch_train_step(policy_log_probs, response_mask, gradient_accumulation_steps, loss_type, ...)` — per-token loss → masked_mean over response → mean over batch → divide by `gradient_accumulation_steps` → backward.

**Deliverable:** implementation. Test: `uv run pytest -k test_grpo_microbatch_train_step`

Implemented in [`cs336_alignment/grpo.py`](../cs336_alignment/grpo.py) — `grpo_microbatch_train_step`.

### Problem `grpo_train_loop` — 5 points

Full GRPO training loop. Default hyperparameters: `n_grpo_steps=200`, `lr=1e-5`, `rollout_batch_size=256`, `group_size=8`, `gradient_accumulation_steps=128`, `loss_type=reinforce_with_baseline`.

**Deliverable:** training script + validation reward curve + sample rollouts over time.

Implemented in [`cs336_alignment/train_grpo.py`](../cs336_alignment/train_grpo.py). Logs to wandb (`ece405-grpo` project).

Run on Koa:
```bash
koa submit scripts/run_grpo.sh
```

In [ ]:
# TODO: display validation reward curve from wandb / results dir

---
## Section 8 — GRPO Experiments

### Problem `grpo_learning_rate` — 2 points

Sweep learning rate. **Deliverable:** validation reward curves; model achieving ≥25% MATH accuracy on at least one LR.

In [ ]:
# TODO: LR sweep results

### Problem `grpo_baselines` — 2 points

Compare `no_baseline` vs `reinforce_with_baseline`. **Deliverable:** reward curves and commentary.

In [ ]:
# TODO: baseline ablation results

### Problem `think_about_length_normalization` — 1 point

Compare `masked_mean` vs `masked_normalize` (constant = max generation length). Pros/cons of each?

**Deliverable:** written analysis.

> *(Written response — TODO)*

### Problem `grpo_length_normalization` — 2 points

Empirical: GRPO with `masked_mean` vs `masked_normalize`. Report curves and gradient norm stability.

In [ ]:
# TODO: length normalization ablation results

### Problem `grpo_group_standard_deviation` — 2 points

Compare `use_std_normalization=True` vs `False`. Report curves and stability commentary.

In [ ]:
# TODO: std normalization ablation results

### Problem `grpo_off_policy` — implementation

Off-policy GRPO with multiple epochs per rollout batch, `old_log_probs` computed once, `loss_type="grpo_clip"`.

**Deliverable:** implementation in `cs336_alignment/train_grpo.py` (already wired via `--loss-type grpo_clip --epochs-per-rollout-batch N`).

### Problem `grpo_off_policy_sweep` — 4 points

Fix `rollout_batch_size=256`. Sweep `epochs_per_rollout_batch` × `train_batch_size`. Compare on-policy baseline by validation accuracy and wall-clock.

In [ ]:
# TODO: off-policy sweep results

### Problem `grpo_off_policy_clip_ablation` — 2 points

Add `loss_type="grpo_no_clip"` ($-\pi/\pi_{old} \cdot A_t$). Compare to clipped: entropy, response length, gradient norm.

In [ ]:
# TODO: no-clip ablation results

### Problem `grpo_prompt_ablation` — 2 points

Train with `question_only.prompt` + `question_only_reward_fn`. Compare to R1-Zero prompt: entropy, response length, gradient norm.

In [ ]:
# TODO: prompt ablation results

In [ ]:
# TODO: leaderboard run results

---
## Optional Supplement — Instruction Tuning, MMLU/GSM8K, DPO, Safety

Covers 2024-style problems (ECE405 deviations #4, #6, #7). See `cs336_spring2024_assignment5_alignment.pdf` and `cs336_spring2025_assignment5_supplement_safety_rlhf.pdf`.

### Problem `sft` — Instruction Tuning

Pack Alpaca-style examples into fixed-length sequences; train Qwen2.5-0.5B for ~30 min.

**Deliverable:** `get_packed_sft_dataset`, `run_iterate_batches` in `tests/adapters.py`.

In [ ]:
# TODO

### Problem `mmlu_baseline`

Zero-shot MMLU. Implement `run_parse_mmlu_response` to extract A/B/C/D.

In [ ]:
# TODO

### Problem `gsm8k_baseline`

Zero-shot GSM8K. `run_parse_gsm8k_response` returns the last numeric token.

In [ ]:
# TODO

### Problem `alpaca_eval_baseline`

ECE405 deviation #4: edit `scripts/alpaca_eval_vllm_llama3_70b_fn` so `model_name` points at the local Qwen2.5-3B-Instruct dir.

In [ ]:
# TODO

### Problem `sst_baseline`

Run `scripts/evaluate_safety.py` with local Qwen2.5-3B-Instruct against SimpleSafetyTests.

In [ ]:
# TODO

### Problem `dpo_training`

DPO on Anthropic HH for ~30 min. Single GPU: query reference and trained model consecutively.

**Deliverable:** `run_compute_per_instance_dpo_loss` in `tests/adapters.py`.

In [ ]:
# TODO

---
## Tests

In [37]:
import sys, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
pkg_root = MATH_BASELINE_DIR.parents[1]
if str(pkg_root) not in sys.path:
    sys.path.insert(0, str(pkg_root))
from cs336_alignment.supplement import compute_dpo_loss
FIXTURES = pkg_root / 'tests' / 'fixtures'
tok = AutoTokenizer.from_pretrained('gpt2')
m   = AutoModelForCausalLM.from_pretrained(str(FIXTURES / 'tiny-gpt2'))
mr  = AutoModelForCausalLM.from_pretrained(str(FIXTURES / 'tiny-gpt2-ref'))
prompt, good, bad = 'The quick brown fox jumps over', 'the lazy dog.', 'their crazy frog.'
loss = compute_dpo_loss(lm=m, lm_ref=mr, tokenizer=tok, beta=0.5,
                        prompt=prompt, response_chosen=good, response_rejected=bad)
print(f'Computed: {loss.item():.6f}  Expected: 0.578500')
p_ids, r_ids = tok.encode(prompt), tok.encode(good)
comb = tok.encode(prompt + good)
print(f'prompt ({len(p_ids)}): {p_ids}')
print(f'response ({len(r_ids)}): {r_ids}')
print(f'combined ({len(comb)}): {comb}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/28 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/28 [00:00<?, ?it/s]

Computed: 0.514669  Expected: 0.578500
prompt (6): [464, 2068, 7586, 21831, 18045, 625]
response (4): [1169, 16931, 3290, 13]
combined (10): [464, 2068, 7586, 21831, 18045, 625, 1169, 16931, 3290, 13]


In [35]:
import subprocess, sys

pkg_root = MATH_BASELINE_DIR.parents[1]  # ece405_assignment3/

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest", "transformers"], check=True)

result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-v", "--tb=short", "--no-header"],
    cwd=pkg_root,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
print("Exit code:", result.returncode)

============================= test session starts ==============================
collecting ... collected 31 items

tests/test_data.py::test_packed_sft_dataset PASSED                       [  3%]
tests/test_data.py::test_iterate_batches PASSED                          [  6%]
tests/test_dpo.py::test_per_instance_dpo_loss 
-------------------------------- live log call ---------------------------------
WARNING  huggingface_hub.utils._http:_http.py:904 Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
FAILED                                                                   [  9%]
tests/test_grpo.py::test_compute_group_normalized_rewards_normalize_by_std PASSED [ 12%]
tests/test_grpo.py::test_compute_group_normalized_rewards_no_normalize_by_std PASSED [ 16%]
tests/test_grpo.py::test_compute_naive_policy_gradient_loss PASSED       [ 19%]
tests/test_grpo.py::test_compute_grpo_clip_loss_large_cliprange PAS